In [28]:
from py.block_a import *

In [29]:
def getSkewness(df: pd.DataFrame, experience: str):
    df_exp = df[df['experience'] == experience]
    df_exp = df_exp.dropna()

    mean_salary = (df_exp["salary_to"] + df_exp['salary_from']) / 2

    return mean_salary.skew()
    


In [30]:
skewness_beginner = getSkewness(df, 'Нет опыта')
skewness_beginner

np.float64(7.038004540909262)

In [31]:
skewness_1_3_year = getSkewness(df, 'От 1 года до 3 лет')
skewness_1_3_year

np.float64(10.783978077501924)

In [32]:
skewness_3_6_year = getSkewness(df, 'От 3 до 6 лет')
skewness_3_6_year

np.float64(7.8222608983636235)

In [33]:
skewness_more_6_year = getSkewness(df, 'Более 6 лет')
skewness_more_6_year

np.float64(7.230426151703995)

In [34]:
# B1. Наибольший коэффициент асимметрии у группы <от 1 года до 3 лет>.
# Это может объясняться ошибками парсинга, выбросами, значения которых сильно выше для типичных этой группы.
# Хвост распределения тянется вправо(правосторонняя ассиметрия), так как существуют аномально большие значения
# Так же аномальные значения возможны из-за неверной интерпретации валют, географии рабочего места.

In [35]:
# B2. среднее арифметическое чувствительно к каждому значению, включая экстремальные выбросы, а медиана — нет.
# Медиана — это значение, которое делит выборку пополам, и на неё влияют только значения в середине упорядоченного ряда.
''' Когда распределение скошено вправо (положительная асимметрия), основная масса значений сосредоточена на 
относительно низких зарплатах, но есть небольшой «хвост» из очень высоких зарплат '''

# Чем выше коэффициент асимметрии, тем сильнее расхождение среднего и медианы

' Когда распределение скошено вправо (положительная асимметрия), основная масса значений сосредоточена на \nотносительно низких зарплатах, но есть небольшой «хвост» из очень высоких зарплат '

In [36]:
def getRangeMetrics(df: pd.DataFrame, experience: str):
    df_exp = df[df['experience'] == experience]
    df_exp = df_exp.dropna()

    mean_salary = (df_exp["salary_to"] + df_exp['salary_from']) / 2

    metrics = [
        mean_salary.std(),  # стандартное отклонение
        mean_salary.quantile(0.75) - mean_salary.quantile(0.25), # межквартильный размах
        mean_salary.std() / mean_salary.mean() # коэффициент вариации
    ]

    return metrics


In [37]:
getRangeMetrics(df, 'Нет опыта')

[np.float64(1221015.8103649204),
 np.float64(32500.0),
 np.float64(4.698577010177253)]

In [38]:
getRangeMetrics(df, 'От 1 года до 3 лет')

[np.float64(3121772.9622213817),
 np.float64(110000.0),
 np.float64(5.238005086454967)]

In [39]:
getRangeMetrics(df, 'От 3 до 6 лет')

[np.float64(2759138.8013550737),
 np.float64(100000.0),
 np.float64(4.888030269303432)]

In [40]:
getRangeMetrics(df, 'Более 6 лет')

[np.float64(5193827.6181315435),
 np.float64(167500.0),
 np.float64(4.860643906453196)]

In [41]:
# Данные сильно искажены выбросами, CV > 1 указывает на то, что стандартное отклонение больше среднего. Такое возможно только при наличии экстремальных выбросов.
# Напишем функцию для фильтрации от аномалий

In [42]:
# Подготовка данных: расчёт точечной зарплаты
import numpy as np
def calculate_salary(row):
    salary_from = row['salary_from']
    salary_to = row['salary_to']
    if pd.notna(salary_from) and pd.notna(salary_to):
        if salary_from <= salary_to:
            return (salary_from + salary_to) / 2
        else:
            return (salary_to + salary_from) / 2
    elif pd.notna(salary_from):
        return salary_from
    elif pd.notna(salary_to):
        return salary_to
    else:
        return np.nan

In [43]:
df['salary'] = df.apply(calculate_salary, axis=1)
df = df.dropna(subset=['salary'])

In [44]:
# Очистка от аномалий (ключевой шаг!)
LOW_SALARY = 10**4
HIGH_SALARY = 10**6
df_clean = df[(df['salary'] >= LOW_SALARY) & (df['salary'] <= HIGH_SALARY)]

In [45]:
# Функция расчёта метрик разброса
def getRangeMetrics(df: pd.DataFrame, experience: str):
    df_exp = df[df['experience'] == experience].dropna(subset=['salary'])
    
    if df_exp.empty:
        return [float('nan'), float('nan'), float('nan')]
    
    salary = df_exp['salary']
    std_dev = salary.std()
    iqr = salary.quantile(0.75) - salary.quantile(0.25)
    cv = std_dev / salary.mean() if salary.mean() != 0 else float('nan')
    
    return [std_dev, iqr, cv]

In [46]:
# Проверяем на очищенных данных
metrics = getRangeMetrics(df_clean, 'Нет опыта')
print(f"std: {metrics[0]:.0f}")
print(f"IQR: {metrics[1]:.0f}")
print(f"CV: {metrics[2]:.2f}")

std: 21338
IQR: 35000
CV: 0.27


In [47]:
metrics = getRangeMetrics(df_clean, 'От 1 года до 3 лет')
print(f"std: {metrics[0]:.0f}")
print(f"IQR: {metrics[1]:.0f}")
print(f"CV: {metrics[2]:.2f}")

std: 112054
IQR: 97500
CV: 0.62


In [48]:
metrics = getRangeMetrics(df_clean, 'От 3 до 6 лет')
print(f"std: {metrics[0]:.0f}")
print(f"IQR: {metrics[1]:.0f}")
print(f"CV: {metrics[2]:.2f}")

std: 69153
IQR: 95000
CV: 0.39


In [49]:
metrics = getRangeMetrics(df_clean, 'Более 6 лет')
print(f"std: {metrics[0]:.0f}")
print(f"IQR: {metrics[1]:.0f}")
print(f"CV: {metrics[2]:.2f}")

std: 114061
IQR: 153125
CV: 0.35


In [50]:
# B3. Наибольший относительный разброс для группы <От 1 года до 3 лет> (CV = 0.62) — зарплаты очень неоднородны.
# Это самая неопределённая группа для прогнозирования. При медиане 150.000 реальные предложения могут варьироваться от 60.000 до 300.000
'''
   Чтобы снизить неопределённость, необходимо анализировать зарплаты отдельно по городам типу занятости и др.
   Это сузит диапазон и сделает прогноз точнее.
   Для группы 1–3 года IQR = 97 500.
'''
getStatisticsWithOutNaN(df_clean, 'От 1 года до 3 лет') #207500 - 110000

,Медиана,Среднее,Стандартное отклонение,25-й перцентиль,75-й перцентиль
0,147500.0,180820.886234,112001.85023,110000.0,207500.0


In [51]:
# Это значит, что 50% предложений лежат в диапазоне примерно 110000.0 - 207500.0
# Cтоит ориентироваться на верхнюю границу этого диапазона, если есть конкурентные преимущества (опыт работы, знание инструментов, хорошее портфолио)